# Grain Product Strategy Research Walkthrough

This notebook is a clean narrative version of the research process. It is designed to answer four questions:

1. What did we test first, and why?
2. What did each test reveal about the product?
3. Why did we move from generic models to product-specific strategies?
4. Which final strategy is most defensible for a fund-style presentation?

Products covered:
- Soybeans
- Corn
- Wheat SRW / HRW pair

Important requirement:
- Cargill data must be used where relevant.
- The research explicitly tested and used both Cargill inventory (`cgl_inv`) and Cargill crush/activity (`cgl_crush`) in the physical signal families.

## Research Discipline

The point of the research was not to find the highest number by trying everything. The process was:

1. Start with low-overfit baselines.
2. Test whether signal families matter.
3. Test whether IC/family selection helps or overfits.
4. Test whether regimes explain why a signal works sometimes and fails elsewhere.
5. Test fitted models only as benchmarks, not as preferred final strategies.
6. Use performance analysis to find the actual failure mode.
7. Build the simplest product-specific rule that addresses that failure mode.

Overfit controls used throughout:
- Fixed economic signs where possible.
- Equal family weights where possible.
- Observable regime rules, not date labels.
- No fitted weights in the final preferred soybean/corn/wheat rules.
- Cost-adjusted results included where available.
- Failed tests were kept in the logs instead of hidden.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

## Common Test Ladder

Every product went through the same basic audit before product-specific strategy design.

In [ ]:
test_ladder = pd.DataFrame([
    {
        "Stage": "1. Average all signals",
        "Question": "Does a low-overfit diversified basket work?",
        "Why it matters": "If this works, the product may not need complex selection. If it fails, the edge is diluted."
    },
    {
        "Stage": "2. Equal family weight",
        "Question": "Is the problem column-count dilution across families?",
        "Why it matters": "Each family gets equal influence: price, curve, COT, public physical, Cargill physical."
    },
    {
        "Stage": "3. IC family selection",
        "Question": "Can train/validation IC identify the useful family?",
        "Why it matters": "Useful diagnostic, but high overfit risk if validation IC does not transfer OOS."
    },
    {
        "Stage": "4. Regime best family",
        "Question": "Does the best family depend on trend/MR or volatility regime?",
        "Why it matters": "Many commodity signals work only in some states."
    },
    {
        "Stage": "5. Regime average families",
        "Question": "Can we keep regime logic while reducing selection risk?",
        "Why it matters": "Averaging families is less selection-heavy than picking one winner."
    },
    {
        "Stage": "6. OLS/Kalman",
        "Question": "Can dynamic fitted coefficients adapt better than fixed rules?",
        "Why it matters": "Benchmark only. Coefficient estimation can overfit noisy futures data."
    },
    {
        "Stage": "7. Ridge",
        "Question": "Can regularized fitted coefficients work robustly?",
        "Why it matters": "Ridge reduces but does not remove coefficient overfit."
    },
    {
        "Stage": "8. Performance analyst",
        "Question": "Where does the strategy actually fail?",
        "Why it matters": "This points to the next economic improvement: low-vol control, abundant-supply guard, pair trend awareness, etc."
    },
])
test_ladder

# Soybeans

## Economic Rationale

Soybeans are directly tied to crushing economics, export competitiveness, and crop-weather stress.

Core economic drivers tested:
- Cargill crush pressure: direct soybean processing signal.
- Cargill inventory pressure: physical availability/tightness signal.
- External crush proxy: soybean meal and soybean oil leadership versus soybeans.
- FX/export pressure: USD, BRL, and CNY affect US soybean competitiveness and China demand.
- Weather HDD/CDD/GDD: crop-belt stress can affect supply expectations.

The research question was: can we improve on the provided Cargill soybean signal without introducing fitted weights or overfit-prone model selection?

In [ ]:
soybean_steps = pd.DataFrame([
    {
        "Step": 1,
        "Logical rationale": "Start with provided-data baseline and confirm Cargill value.",
        "Test / formula": "given_crush_pressure = avg(crush_surprise, crush_utilization)",
        "Result": "OOS Sharpe 1.604; PnL 207; DD -95",
        "Decision": "Keep. Cargill crush is a real soybean signal."
    },
    {
        "Step": 2,
        "Logical rationale": "Test whether broader provided signals improve robustness.",
        "Test / formula": "given_conservative = trend + inventory + cgl_inv + cgl_crush",
        "Result": "OOS Sharpe 1.393; PnL 227; DD -88",
        "Decision": "Useful baseline, but not best."
    },
    {
        "Step": 3,
        "Logical rationale": "Test fitted dynamic weights as a benchmark.",
        "Test / formula": "OLS/Kalman dynamic linear coefficients",
        "Result": "OOS Sharpe 0.357; PnL 369; DD -348",
        "Decision": "Reject as final. Did not beat simple economic blends and adds coefficient risk."
    },
    {
        "Step": 4,
        "Logical rationale": "Test whether external soybean economics add value.",
        "Test / formula": "0.5 * Cargill crush + 0.5 * weather HDD/CDD",
        "Result": "OOS Sharpe 2.639; PnL 154; DD -34",
        "Decision": "Keep as best Sharpe/DD evidence."
    },
    {
        "Step": 5,
        "Logical rationale": "Test broader but still economic external blend.",
        "Test / formula": "40% physical + 20% FX/export + 20% external crush + 20% weather",
        "Result": "OOS Sharpe 2.168; PnL 222; DD -55",
        "Decision": "Recommended base: better fund-style balance."
    },
    {
        "Step": 6,
        "Logical rationale": "Key-period analysis showed weakness in low-vol / abundant-supply periods.",
        "Test / formula": "Performance analyst by period",
        "Result": "Low-price abundant supply PnL -38, DD -71; trade war PnL -14, DD -48",
        "Decision": "Need risk control, not more alpha fitting."
    },
    {
        "Step": 7,
        "Logical rationale": "Add simple observable low-vol risk control without fitting weights.",
        "Test / formula": "If 60d vol < 0.8 * expanding median, cut exposure by half",
        "Result": "OOS Sharpe 2.474; PnL 162; DD -35",
        "Decision": "Final fund-style soybean version."
    },
])
soybean_steps

## Soybean Final Pseudo-Code

In [ ]:
# Soybean final fund-style logic, expressed as pseudo-code.

soybean_pseudocode = """
given_physical_family = avg(
    -public_inventory_change,
    -receipts_change,
    -cgl_inventory_change,
    crush_surprise,
    crush_utilization,
    curve_tightness
)

external_fx_export_family = avg(
    -USD_pressure,
    -BRL_weakness_pressure,
    -CNY_weakness_pressure
)

external_crush_family = avg(
    soybean_meal_leadership,
    soybean_oil_leadership,
    external_crush_margin_proxy
)

weather_family = avg(
    CDD_HDD_GDD_stress,
    heat_stress,
    dryness,
    dry_CDD_interaction,
    planting_precipitation,
    harvest_freeze_stress
)

base_signal = (
    0.40 * given_physical_family
  + 0.20 * external_fx_export_family
  + 0.20 * external_crush_family
  + 0.20 * weather_family
)

smoothed_signal = EWMA(base_signal, halflife=2)
position = vol_target(smoothed_signal, lagged_60d_soybean_pnl_vol)
position = clip(position, 0, 0.50)

low_vol = soybean_60d_pnl_vol < 0.80 * expanding_median(soybean_60d_pnl_vol)
if low_vol:
    position = 0.50 * position
"""
print(soybean_pseudocode)

## Soybean Interpretation

The strongest message is not simply that weather improved the backtest. The stronger research message is:

1. Cargill crush was already a strong provided-data signal.
2. Broad/fitted models did not beat simple economic blends.
3. External weather and crush/FX signals improved the economic completeness of the model.
4. The remaining weakness was not solved by another alpha model; it was solved by a simple low-vol risk control.

Recommended soybean strategy:
- Use the drawdown-priority economic blend as the base.
- Apply the low-vol half-exposure rule for fund-style drawdown control.

# Corn

## Economic Rationale

Corn was harder than soybean because the generic provided-data signals were weak. Corn has a more specific demand channel: ethanol. That made EIA ethanol production/stocks worth testing.

Core economic drivers tested:
- Provided price/trend and physical/Cargill data.
- EIA ethanol production and stocks as corn-demand pressure.
- FX/export pressure.
- Volatility regime behavior.
- Abundant-supply drawdown control.

The research question became: can corn be made fund-usable by combining corn-specific demand data with observable risk control?

In [ ]:
corn_steps = pd.DataFrame([
    {
        "Step": 1,
        "Logical rationale": "Start with simple provided-data/family tests to see if generic grain signals work.",
        "Test / formula": "Avg/equal family provided signals",
        "Result": "Avg OOS Sharpe -0.223, DD -437; equal-family Sharpe -0.400, DD -534",
        "Decision": "Reject. Generic corn signal is weak."
    },
    {
        "Step": 2,
        "Logical rationale": "Test fitted dynamic weights as a benchmark.",
        "Test / formula": "OLS/Kalman dynamic linear coefficients",
        "Result": "OOS Sharpe -0.390; PnL -349; DD -844",
        "Decision": "Reject. Fitted model added risk and did not find robust edge."
    },
    {
        "Step": 3,
        "Logical rationale": "Corn has a unique demand driver, so test ethanol.",
        "Test / formula": "EIA ethanol family",
        "Result": "OOS Sharpe 0.653; PnL 435; DD -333",
        "Decision": "Keep as relevant, but not standalone."
    },
    {
        "Step": 4,
        "Logical rationale": "Blend ethanol conservatively with provided signals.",
        "Test / formula": "80% given_conservative + 10% ethanol + 10% FX/export",
        "Result": "OOS Sharpe 0.353; PnL 203; DD -426",
        "Decision": "Requirement-compliant, but too weak for final."
    },
    {
        "Step": 5,
        "Logical rationale": "Test whether corn needs regime conditioning.",
        "Test / formula": "Vol-regime IC strategy",
        "Result": "OOS Sharpe 0.759; PnL 224; DD -243",
        "Decision": "Best early corn base."
    },
    {
        "Step": 6,
        "Logical rationale": "Key-period analysis showed corn's main failure is abundant-supply drawdown.",
        "Test / formula": "Performance analyst",
        "Result": "Low-price abundant supply DD -416",
        "Decision": "Need observable abundant-supply guard."
    },
    {
        "Step": 7,
        "Logical rationale": "Add no-fit risk control.",
        "Test / formula": "If price below MA or negative momentum, go flat",
        "Result": "OOS Sharpe 1.834; PnL 266; DD -141; full DD -308",
        "Decision": "Best corn version, but use as small satellite sleeve."
    },
])
corn_steps

## Corn Final Pseudo-Code

In [ ]:
corn_pseudocode = """
given_inventory_pressure = avg(
    -public_inventory_change,
    -receipts_change,
    -cgl_inventory_change
)

cgl_crush_activity = avg(
    crush_surprise,
    crush_utilization
)

given_physical_family = avg(
    given_inventory_pressure,
    curve_tightness,
    0.25 * cgl_crush_activity
)

given_price_family = avg(mom_20, mom_60, rev_5)
given_trend = avg(mom_20, mom_60, curve_spread, cot_pm_oi_level)

given_conservative_blend = (
    0.40 * given_physical_family
  + 0.30 * given_trend
  + 0.30 * given_price_family
)

external_ethanol_family = avg(
    ethanol_production_change_4w,
    ethanol_prod_to_stocks,
    ethanol_demand_pressure,
    -ethanol_stocks_change_4w
)

external_fx_export_family = avg(-USD_pressure, -BRL_pressure, -CNY_pressure)

base_signal = 0.80 * given_conservative_blend + 0.10 * external_ethanol_family + 0.10 * external_fx_export_family

# Improved corn version uses vol-regime IC base plus abundant-supply guard.
abundant_supply_guard = (corn_price < corn_252d_moving_average) or (corn_mom_60 < 0)
if abundant_supply_guard:
    position = 0
else:
    position = vol_target(EWMA(base_signal, halflife=2), lagged_60d_corn_pnl_vol)
    position = clip(position, 0, 0.50)
"""
print(corn_pseudocode)

## Corn Interpretation

Corn is not as strong as soybean or wheat. The useful research finding is that corn should not be forced into the same generic model.

What the tests showed:

1. Generic provided-data averages failed.
2. OLS/Kalman and Ridge-style fitting did not create robust edge.
3. Ethanol mattered economically, but was unstable alone.
4. Volatility regime helped more than trend/MR regime.
5. The main failure mode was low-price abundant-supply drawdown.
6. An observable abundant-supply flat guard materially improved drawdown.

Recommended corn role:
- Small satellite sleeve.
- Use corn-specific demand and regime/risk control.
- Do not present corn as the strongest standalone allocation.

# Wheat SRW / HRW

## Economic Rationale

Directional wheat strategies were weak. The key research shift was to stop treating SRW and HRW as separate outright alphas and instead trade the relative value between them.

Economic intuition:
- SRW and HRW share broad wheat beta, but differ in regional supply/demand and quality dynamics.
- A pair trade removes much of the broad grain beta.
- Price mean reversion in the SRW/HRW relationship was stronger than outright directional wheat prediction.
- Cargill physical data helps anchor relative physical pressure.

The research question became: can wheat be traded as a lower-overfit SRW/HRW spread using simple MR + physical pressure?

In [ ]:
wheat_steps = pd.DataFrame([
    {
        "Step": 1,
        "Logical rationale": "First check whether standalone directional wheat works.",
        "Test / formula": "Best standalone SRW and HRW directional sleeves",
        "Result": "SRW OOS Sharpe -0.044, DD -383; HRW Sharpe -0.044, DD -283",
        "Decision": "Reject standalone wheat."
    },
    {
        "Step": 2,
        "Logical rationale": "Family/regime directional tests were still weak.",
        "Test / formula": "Best directional regime/family rows",
        "Result": "SRW Sharpe 0.184, DD -200; HRW Sharpe 0.030, DD -197",
        "Decision": "Directional wheat not enough."
    },
    {
        "Step": 3,
        "Logical rationale": "Test fitted dynamic weights as a benchmark.",
        "Test / formula": "OLS/Kalman dynamic linear coefficients",
        "Result": "SRW OOS Sharpe 0.064, DD -441; HRW OOS Sharpe -0.554, DD -719",
        "Decision": "Reject. Directional fitted models were not robust."
    },
    {
        "Step": 4,
        "Logical rationale": "Wheat economics suggest relative value between SRW and HRW.",
        "Test / formula": "90% SRW/HRW price MR + 10% Cargill physical",
        "Result": "OOS Sharpe 1.129; PnL 26.7; DD -19.9; full Sharpe 1.403",
        "Decision": "Recommended clean wheat base."
    },
    {
        "Step": 5,
        "Logical rationale": "Check if pair MR struggles in trend-like wheat periods.",
        "Test / formula": "Performance analyst",
        "Result": "2019 floods PnL -11.6; COVID recovery PnL -14.4",
        "Decision": "Add small trend awareness."
    },
    {
        "Step": 6,
        "Logical rationale": "Add trend signal without fitted coefficients.",
        "Test / formula": "80% MR/Cargill + 20% pair trend",
        "Result": "OOS Sharpe 1.291; PnL 36.4; DD -29.4",
        "Decision": "Best trend-aware wheat variant."
    },
    {
        "Step": 7,
        "Logical rationale": "Test conservative risk-control alternative.",
        "Test / formula": "Flat when trend conflicts with MR",
        "Result": "OOS Sharpe 2.145; PnL 26.7; DD -17.0",
        "Decision": "Best Sharpe/DD, but more timing-like; present as conservative overlay."
    },
])
wheat_steps

## Wheat Final Pseudo-Code

In [ ]:
wheat_pseudocode = """
# Build pair signals from SRW minus HRW families.

pair_price_mr = SRW_rev_5 - HRW_rev_5

pair_physical_cargill = avg(
    SRW_cgl_inventory_pressure - HRW_cgl_inventory_pressure,
    SRW_crush_surprise - HRW_crush_surprise,
    SRW_crush_utilization - HRW_crush_utilization
)

base_pair_signal = 0.90 * pair_price_mr + 0.10 * pair_physical_cargill

# Trend-aware variant:
ratio = SRW_adjusted_price / HRW_adjusted_price
pair_trend = avg(
    zscore(ratio.pct_change(20)),
    zscore(ratio.pct_change(60))
)

trend_aware_signal = 0.80 * base_pair_signal + 0.20 * pair_trend

# Conservative overlay:
trend_state = abs(pair_trend) > expanding_median(abs(pair_trend))
trend_conflict = trend_state and sign(pair_trend) != sign(base_pair_signal)

if trend_conflict:
    pair_position = 0
else:
    pair_position = vol_target(EWMA(base_pair_signal, halflife=5), lagged_pair_pnl_vol)

# Positive pair_position: long SRW, short HRW.
# Negative pair_position: short SRW, long HRW.
"""
print(wheat_pseudocode)

## Wheat Interpretation

The wheat result is the cleanest example of research iteration:

1. Directional SRW failed.
2. Directional HRW failed.
3. Directional IC/regime/fitted models were still weak.
4. The economic structure suggested a pair trade.
5. SRW/HRW pair mean reversion with a small Cargill physical anchor produced a much cleaner OOS profile.
6. A small pair-trend overlay improved trend-like weak periods.

Recommended wheat strategy:
- Base: `90% pair price MR + 10% Cargill physical`.
- Trend-aware variant: `80% MR/Cargill + 20% pair trend`.
- Conservative diagnostic: trend-conflict flat filter.

# Final Strategy Comparison

In [ ]:
final_recommendations = pd.DataFrame([
    {
        "Product": "Soybeans",
        "Recommended strategy": "Drawdown-priority economic blend with low-vol half exposure",
        "Key result": "OOS Sharpe 2.474; PnL 162; DD -35",
        "Why it is defensible": "Uses Cargill physical/crush, external crush, FX/export, and weather; final risk control is observable and no-fit.",
        "Role": "Core product sleeve"
    },
    {
        "Product": "Corn",
        "Recommended strategy": "Vol-regime corn sleeve with abundant-supply flat guard",
        "Key result": "OOS Sharpe 1.834; PnL 266; DD -141; full DD -308",
        "Why it is defensible": "Generic corn models failed; ethanol is economically relevant; abundant-supply guard targets the actual drawdown problem.",
        "Role": "Small satellite sleeve"
    },
    {
        "Product": "Wheat SRW/HRW",
        "Recommended strategy": "SRW/HRW pair MR + Cargill physical; optional 20% pair trend",
        "Key result": "Base OOS Sharpe 1.129, DD -20; trend-aware OOS Sharpe 1.291, DD -29",
        "Why it is defensible": "Outright wheat failed; pair trade removes broad wheat beta and matches SRW/HRW relative-value economics.",
        "Role": "Core diversifying relative-value sleeve"
    },
])
final_recommendations

# What To Prune From The Main Presentation

The following tests should stay in logs/appendix rather than the main story:

- Broad average/equal-family tests after the first baseline slide.
- OLS/Kalman/Ridge as final candidates.
- IC validation winners that failed OOS.
- Broad external equal-weight blends that diluted the edge.
- Low-vol replacement alpha sleeves for soybean that worsened drawdown.
- Standalone directional SRW/HRW wheat strategies.
- Hard wheat trend switches.

They are still useful because they prove what was rejected. But the main presentation should focus on the tests that changed the research direction.

In [ ]:
prune_table = pd.DataFrame([
    {"Product": "Soybeans", "Prune from main story": "OLS/Kalman/Ridge, broad external equal weight, low-vol MR replacement sleeves", "Reason": "They did not beat fixed economic blends or worsened DD."},
    {"Product": "Corn", "Prune from main story": "Flat IC selection, trend/MR regime, OLS/Kalman/Ridge as final", "Reason": "Generic/selection/fitted methods were weak; corn needed ethanol plus risk guard."},
    {"Product": "Wheat", "Prune from main story": "Standalone SRW/HRW, directional IC/regime, hard trend switches", "Reason": "The real discovery was SRW/HRW pair relative value."},
])
prune_table

# Presentation Storyline

A clean way to present the research:

1. I began with low-overfit baselines: average all signals and equal-family weights.
2. Those baselines showed that generic grain signals were diluted, especially for corn and wheat.
3. I tested IC, regime, OLS/Kalman, and Ridge as systematic alternatives.
4. The fitted models and many IC-selected rows had overfit risk or weak OOS behavior.
5. Performance analysis revealed product-specific failure modes:
   - Soybeans: weak in low-vol / abundant-supply periods.
   - Corn: poor abundant-supply drawdown and weak generic signal quality.
   - Wheat: outright directional wheat did not work; relative value was better.
6. I then built lower-overfit product-specific strategies:
   - Soybeans: physical + FX/export + external crush + weather, with low-vol half exposure.
   - Corn: ethanol/vol-regime idea with abundant-supply flat guard.
   - Wheat: SRW/HRW pair mean reversion plus Cargill physical, optionally trend-aware.

Final message:

The strongest strategies were not the most complex fitted models. They were the simplest product-specific economic rules that directly addressed the failure mode discovered in the backtests.

# Source Files

Primary experiment/code files:

- `family_regime_model_comparison.py`
- `linear_online_model_experiment.py`
- `soybean_external_signal_experiment.py`
- `soybean_low_vol_switch_experiment.py`
- `corn_external_signal_experiment.py`
- `corn_abundant_supply_improvement.py`
- `wheat_improvement_experiment.py`

Primary notes:

- `notes/soybeans.txt`
- `notes/soybean_low_vol_switch.txt`
- `notes/corn.txt`
- `notes/corn_abundant_supply_improvement.txt`
- `notes/wheat_improvement.txt`
- `notes/family_regime_model_comparison.txt`
- `notes/linear_online_models_corn_soybean.txt`
- `notes/strategy_experiment_log.txt`